In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import copy
import requests

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
from torchvision import transforms as TF
import torch.nn.functional as F
from torch.optim import AdamW

from transformers import SegformerForSemanticSegmentation, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import jaccard_score
from tqdm import tqdm
from PIL import Image

In [2]:
class BDDDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        mask_path = self.masks[idx]
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert('L')  # Convert mask to grayscale

        # Convert mask to binary format with 0 and 1 values
        mask = np.array(mask) 
        mask = (mask > 0).astype(np.uint8)  # Assuming non-zero pixels are lanes

        # Convert to PIL Image for consistency in transforms
        mask = Image.fromarray(mask)

        if self.transform:
            image = self.transform(image)
        mask = TF.functional.resize(img=mask, size=[360, 640], interpolation=Image.NEAREST)
        mask = TF.functional.to_tensor(mask)
        mask = (mask > 0).long()  # Threshold back to binary and convert to LongTensor

        return image, mask

In [3]:
# Gather all images and masks
images_dir = '/kaggle/input/sr-coal-dataset/coal_sr_bigdataset/new_images'
masks_dir = '/kaggle/input/sr-coal-dataset/coal_sr_bigdataset/new_masks'
images = [os.path.join(images_dir, img) for img in os.listdir(images_dir) if img.endswith('.png')]
masks = [os.path.join(masks_dir, img.replace('.bmp', '.png')) for img in os.listdir(images_dir) if img.endswith('.png')]

# Split the dataset
train_images, valid_images, train_masks, valid_masks = train_test_split(images, masks, test_size=0.2, random_state=42)


In [4]:
# Define the appropriate transformations
transform = TF.Compose([
    TF.Resize((360, 640)),
    TF.ToTensor(),
    TF.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create the datasets
train_dataset = BDDDataset(images=train_images, masks=train_masks, transform=transform)
valid_dataset = BDDDataset(images=valid_images, masks=valid_masks, transform=transform)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, num_workers=4)


In [5]:
# Load the pre-trained model
model = SegformerForSemanticSegmentation.from_pretrained('nvidia/segformer-b2-finetuned-ade-512-512')

# Adjust the number of classes for BDD dataset
model.config.num_labels = 2  # Replace with the actual number of classes

config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [6]:
# Check for CUDA acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device);

In [7]:
def mean_iou(preds, labels, num_classes):
    # Flatten predictions and labels
    preds_flat = preds.view(-1)
    labels_flat = labels.view(-1)

    # Check that the number of elements in the flattened predictions
    # and labels are equal
    if preds_flat.shape[0] != labels_flat.shape[0]:
        raise ValueError(f"Predictions and labels have mismatched shapes: "
                         f"{preds_flat.shape} vs {labels_flat.shape}")

    # Calculate the Jaccard score for each class
    iou = jaccard_score(labels_flat.cpu().numpy(), preds_flat.cpu().numpy(),
                        average=None, labels=range(num_classes))

    # Return the mean IoU
    return np.mean(iou)

In [8]:
# Define the optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

# Define the learning rate scheduler
num_epochs = 15
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

# Placeholder for best mean IoU and best model weights
best_iou = 0.0
best_model_wts = copy.deepcopy(model.state_dict())


In [9]:
for epoch in range(num_epochs):
    model.train()
    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch")
    for batch in train_iterator:
        images, masks = batch
        images = images.to(device)
        masks = masks.to(device).long()  # Ensure masks are LongTensors
 
        # Remove the channel dimension from the masks tensor
        masks = masks.squeeze(1)  # This changes the shape from [batch, 1, H, W] to [batch, H, W]
        optimizer.zero_grad()
 
        # Pass pixel_values and labels to the model
        outputs = model(pixel_values=images, labels=masks,return_dict=True)
         
        loss = outputs["loss"]
        loss.backward()
 
        optimizer.step()
        lr_scheduler.step()
        outputs = F.interpolate(outputs["logits"], size=masks.shape[-2:], mode="bilinear", align_corners=False)
         
        train_iterator.set_postfix(loss=loss.item())
        


Epoch 15/15: 100%|██████████| 1498/1498 [46:00<00:00,  1.84s/batch, loss=0.00477]


In [10]:
# Evaluation loop for each epoch
model.eval()
total_iou = 0
num_batches = 0
valid_iterator = tqdm(valid_loader, desc="Validation", unit="batch")
for batch in valid_iterator:
    images, masks = batch
    images = images.to(device)
    masks = masks.to(device).long()
 
    with torch.no_grad():
        # Get the logits from the model and apply argmax to get the predictions
        outputs = model(pixel_values=images,return_dict=True)
        outputs = F.interpolate(outputs["logits"], size=masks.shape[-2:], mode="bilinear", align_corners=False)
        preds = torch.argmax(outputs, dim=1)
        preds = torch.unsqueeze(preds, dim=1)
 
    preds = preds.view(-1)
    masks = masks.view(-1)
 
    # Compute IoU
    iou = mean_iou(preds, masks, model.config.num_labels)
    total_iou += iou
    num_batches += 1
    # Print IoU for current batch
    print(f"Batch {num_batches}/{len(valid_loader)} - IoU: {iou:.4f}")
    valid_iterator.set_postfix(mean_iou=iou)
 
epoch_iou = total_iou / num_batches
print(f"Epoch {epoch+1}/{num_epochs} - Mean IoU: {epoch_iou:.4f}")
 
# Check for improvement
if epoch_iou > best_iou:
    print(f"Validation IoU improved from {best_iou:.4f} to {epoch_iou:.4f}")
    best_iou = epoch_iou
    best_model_wts = copy.deepcopy(model.state_dict())
    torch.save(best_model_wts, 'best_model.pth')

Validation:   0%|          | 1/375 [00:02<13:13,  2.12s/batch, mean_iou=0.997]

Batch 1/375 - IoU: 0.9968


Validation:   1%|          | 2/375 [00:03<11:18,  1.82s/batch, mean_iou=0.997]

Batch 2/375 - IoU: 0.9970


Validation:   1%|          | 3/375 [00:05<10:44,  1.73s/batch, mean_iou=0.997]

...

Batch 373/375 - IoU: 0.9962


Validation: 100%|█████████▉| 374/375 [05:53<00:00,  1.09batch/s, mean_iou=0.994]

Batch 374/375 - IoU: 0.9940


Validation: 100%|██████████| 375/375 [05:53<00:00,  1.06batch/s, mean_iou=0.996]

Batch 375/375 - IoU: 0.9965
Epoch 15/15 - Mean IoU: 0.9968
Validation IoU improved from 0.0000 to 0.9968


In [11]:
# Load the trained model 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model1 = SegformerForSemanticSegmentation.from_pretrained('nvidia/segformer-b2-finetuned-ade-512-512')
 
# Replace with the actual number of classes
model1.config.num_labels = 2 
 
# Load the state from the fine-tuned model and set to model.eval() mode
model1.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model1.to(device)
model1.eval()

SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [12]:
# Perform transformations
data_transforms = TF.Compose([
    TF.ToPILImage(),
    TF.Resize((360, 640)),
    TF.ToTensor(),
    TF.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [13]:
# Load and preprocess the image
image_path = '/kaggle/input/sr-coal-dataset/coal_sr_bigdataset/new_masks/10_patch_10.png'
image = cv2.imread(image_path)
input_tensor = data_transforms(image).unsqueeze(0).to(device)

# Inference
with torch.no_grad():
    outputs = model1(pixel_values=input_tensor, return_dict=True)
    outputs = F.interpolate(outputs["logits"], size=(360, 640), mode="bilinear", align_corners=False)
    preds = torch.argmax(outputs, dim=1)
    preds = torch.unsqueeze(preds, dim=1)
    predicted_mask = (torch.sigmoid(preds) > 0.5).float()

# Create an RGB version of the mask to overlay on the original image
mask_np = predicted_mask.cpu().squeeze().numpy()
mask_resized = cv2.resize(mask_np, (image.shape[1], image.shape[0]))

# Modify this section to create a green mask
mask_rgb = np.zeros((mask_resized.shape[0], mask_resized.shape[1], 3), dtype=np.uint8)
mask_rgb[:, :, 1] = (mask_resized * 255).astype(np.uint8)  # Set only the green channel

# Post-processing for mask smoothening
# Remove noise
kernel = np.ones((3,3), np.uint8)
opening = cv2.morphologyEx(mask_rgb, cv2.MORPH_OPEN, kernel, iterations=2)

# Close small holes
closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=2)

# Overlay the mask on the image
blended = cv2.addWeighted(image, 0.65, closing, 0.6, 0)

# Save the result
cv2.imwrite('output_image.jpg', blended)
print("Inference completed. Output saved as 'output_image.jpg'")

Inference completed. Output saved as 'output_image.jpg'


In [14]:
#!zip -r file_2.zip /kaggle/working/processed_5e4
